# Incubators in India — Agentic URL Validation Pipeline

A fully autonomous multi-agent pipeline that:
1. **Fetches URLs** via Tavily Search API across 6 curated queries
2. **Validates content** — extracts text, hashes, filters stubs
3. **Scores relevance** via `meta/meta-llama-3-8b-instruct` on Replicate (zero-shot)
4. **Deduplicates** using FAISS + HDBSCAN semantic clustering
5. **Self-critiques** via Eval Agent and logs reflections
6. **Archives** final results to SQLite + FAISS index

---
**Requirements:**
- `TAVILY_API_KEY` — get free at https://tavily.com
- `REPLICATE_API_TOKEN` — get free at https://replicate.com

> Set both as Colab Secrets (lock icon in left sidebar) or paste directly in Cell 2.

## Cell 1 — Install dependencies

In [1]:
%%capture

!pip install tavily-python replicate sentence-transformers faiss-cpu hdbscan langgraph langchain-core requests beautifulsoup4 lxml

## Cell 2 — API keys

In [2]:
import os

# Option A: Colab Secrets (recommended)
try:
    from google.colab import userdata
    os.environ['TAVILY_API_KEY']      = userdata.get('TAVILY_API_KEY')
    os.environ['REPLICATE_API_TOKEN'] = userdata.get('REPLICATE_API_TOKEN')
    print('Keys loaded from Colab Secrets.')
except Exception:
    # Option B: paste directly
    os.environ['TAVILY_API_KEY']      = 'tvly-XXXXXXXXXXXXXXXX'  # <-- replace
    os.environ['REPLICATE_API_TOKEN'] = 'r8_XXXXXXXXXXXXXXXX'    # <-- replace
    print('Keys loaded from inline values.')

assert os.environ.get('TAVILY_API_KEY'),      'TAVILY_API_KEY missing!'
assert os.environ.get('REPLICATE_API_TOKEN'), 'REPLICATE_API_TOKEN missing!'
print('All keys present.')

Keys loaded from Colab Secrets.
All keys present.


## Cell 3 — Imports & config

In [3]:
import os, json, uuid, time, hashlib, sqlite3, re
from dataclasses import dataclass, field
from typing import List, Optional

import numpy as np
import requests
from bs4 import BeautifulSoup
import replicate
from tavily import TavilyClient
from sentence_transformers import SentenceTransformer
import faiss
import hdbscan

REPLICATE_MODEL      = 'meta/meta-llama-3-8b-instruct'
RELEVANCE_THRESHOLD  = 0.70
ACCURACY_PAUSE       = 0.60
ACCURACY_RERUN       = 0.85
MIN_WORD_COUNT       = 100
TAVILY_MAX_RESULTS   = 15
MAX_QUERY_VARIANTS   = 10
DB_PATH              = 'pipeline.db'

print('Config loaded.')
print(f'Model                : {REPLICATE_MODEL}')
print(f'Relevance threshold  : {RELEVANCE_THRESHOLD}')
print(f'Max Tavily results   : {TAVILY_MAX_RESULTS} per generated query')
print(f'Generated query size : {MAX_QUERY_VARIANTS}')

Config loaded.
Model                : meta/meta-llama-3-8b-instruct
Relevance threshold  : 0.7
Max Tavily results   : 15 per generated query
Generated query size : 10


/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


In [4]:
# Runtime query input (edit this cell for each run)
DEFAULT_USER_QUERY = 'INSOMNIA by Christopher Nolan'

USER_QUERY = os.environ.get('USER_QUERY', DEFAULT_USER_QUERY).strip()
USER_REGION = os.environ.get('USER_REGION', '').strip()
RUN_ID = str(uuid.uuid4())

if not USER_QUERY:
    raise ValueError('USER_QUERY is required.')

print('=' * 60)
print(f'  USER_QUERY : {USER_QUERY}')
print(f'  USER_REGION: {USER_REGION if USER_REGION else "<none>"}')
print(f'  RUN_ID     : {RUN_ID}')
print('=' * 60)

  USER_QUERY : INSOMNIA by Christopher Nolan
  USER_REGION: <none>
  RUN_ID     : 6bada1e3-6f47-4cf3-a1e1-378128941e56


## Cell 4 — Task schema

In [5]:
@dataclass
class Task:
    id:       str  = field(default_factory=lambda: str(uuid.uuid4()))
    status:   str  = 'pending'
    data:     dict = field(default_factory=dict)
    metadata: dict = field(default_factory=dict)
    requires: List[str] = field(default_factory=list)
    produces: List[str] = field(default_factory=list)

@dataclass
class URLRecord:
    url:              str
    title:            str   = ''
    content:          str   = ''
    content_hash:     str   = ''
    word_count:       int   = 0
    language:         str   = 'en'
    query_source:     str   = ''
    tavily_score:     float = 0.0
    relevance_score:  float = 0.0
    reasoning:        str   = ''
    keywords:         List[str] = field(default_factory=list)
    page_title:       str   = ''
    meta_description: str   = ''
    meta_keywords:    List[str] = field(default_factory=list)
    site_name:        str   = ''
    canonical_url:    str   = ''
    query_fit:        bool  = False
    region_match:     bool  = True
    category:         str   = 'other'
    is_relevant:      bool  = False
    cluster_id:       int   = -1
    is_duplicate:     bool  = False
    embedding:        Optional[List[float]] = None

print('Schemas defined.')

Schemas defined.


## Cell 5 — Memory agent (SQLite)

In [6]:
class MemoryAgent:
    def __init__(self, db_path=DB_PATH):
        self.conn = sqlite3.connect(db_path)
        self._init_db()

    def _migrate_archived_urls_if_needed(self):
        row = self.conn.execute(
            "SELECT sql FROM sqlite_master WHERE type='table' AND name='archived_urls'"
        ).fetchone()
        create_sql = (row[0] or '').lower() if row else ''

        # Legacy schema had url UNIQUE, which collapses rows across runs.
        if 'url text unique' not in create_sql:
            return

        self.conn.executescript("""
            CREATE TABLE IF NOT EXISTS archived_urls_new (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                url TEXT,
                title TEXT,
                score REAL,
                category TEXT,
                keywords TEXT,
                cluster_id INTEGER,
                is_duplicate INTEGER DEFAULT 0,
                query_source TEXT,
                canonical_url TEXT,
                run_id TEXT,
                user_query TEXT,
                run_at TEXT,
                tavily_score REAL,
                archived_at TEXT
            );

            INSERT INTO archived_urls_new
                (url,title,score,category,keywords,cluster_id,is_duplicate,query_source,canonical_url,run_id,user_query,run_at,tavily_score,archived_at)
            SELECT
                url,
                title,
                score,
                category,
                keywords,
                cluster_id,
                0,
                query_source,
                '',
                'legacy_' || id,
                'legacy',
                COALESCE(archived_at, datetime('now')),
                tavily_score,
                archived_at
            FROM archived_urls;

            DROP TABLE archived_urls;
            ALTER TABLE archived_urls_new RENAME TO archived_urls;
        """)
        self.conn.commit()
        print('   [memory] Migrated archived_urls schema to run-scoped format.')

    def _ensure_columns(self):
        existing = {
            row[1] for row in self.conn.execute("PRAGMA table_info(archived_urls)").fetchall()
        }
        required = {
            'is_duplicate': 'INTEGER DEFAULT 0',
            'canonical_url': 'TEXT',
            'run_id': 'TEXT',
            'user_query': 'TEXT',
            'run_at': 'TEXT',
            'archived_at': 'TEXT',
        }
        for col, col_type in required.items():
            if col not in existing:
                self.conn.execute(f"ALTER TABLE archived_urls ADD COLUMN {col} {col_type}")

    def _normalize_legacy_rows(self):
        self.conn.execute(
            "UPDATE archived_urls SET cluster_id = (1000000 + id) WHERE cluster_id IS NULL OR cluster_id < 0"
        )
        self.conn.execute(
            "UPDATE archived_urls SET run_id = ('legacy_' || id) WHERE run_id IS NULL OR run_id = ''"
        )
        self.conn.execute(
            "UPDATE archived_urls SET user_query = 'legacy' WHERE user_query IS NULL OR user_query = ''"
        )
        self.conn.execute(
            "UPDATE archived_urls SET run_at = COALESCE(run_at, archived_at, datetime('now')) WHERE run_at IS NULL OR run_at = ''"
        )

    def _init_db(self):
        self.conn.executescript("""
            CREATE TABLE IF NOT EXISTS routing_log (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                ts TEXT, agent TEXT, decision TEXT, reasoning TEXT
            );

            CREATE TABLE IF NOT EXISTS reflection_log (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                ts TEXT, agent TEXT, issue TEXT, fix TEXT
            );

            CREATE TABLE IF NOT EXISTS archived_urls (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                url TEXT,
                title TEXT,
                score REAL,
                category TEXT,
                keywords TEXT,
                cluster_id INTEGER,
                is_duplicate INTEGER DEFAULT 0,
                query_source TEXT,
                canonical_url TEXT,
                run_id TEXT,
                user_query TEXT,
                run_at TEXT,
                tavily_score REAL,
                archived_at TEXT
            );
        """)
        self._migrate_archived_urls_if_needed()
        self._ensure_columns()
        self._normalize_legacy_rows()
        self.conn.commit()

    def log_routing(self, agent, decision, reasoning=''):
        self.conn.execute(
            'INSERT INTO routing_log (ts,agent,decision,reasoning) VALUES (?,?,?,?)',
            (time.strftime('%H:%M:%S'), agent, decision, reasoning)
        )
        self.conn.commit()

    def log_reflection(self, agent, issue, fix):
        self.conn.execute(
            'INSERT INTO reflection_log (ts,agent,issue,fix) VALUES (?,?,?,?)',
            (time.strftime('%H:%M:%S'), agent, issue, fix)
        )
        self.conn.commit()
        print(f'   [reflection] {agent}: {issue[:80]}')

    def archive_url(self, rec: URLRecord, run_id: str, user_query: str):
        try:
            now = time.strftime('%Y-%m-%d %H:%M:%S')
            self.conn.execute(
                'INSERT INTO archived_urls '
                '(url,title,score,category,keywords,cluster_id,is_duplicate,query_source,canonical_url,run_id,user_query,run_at,tavily_score,archived_at) '
                'VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)',
                (
                    rec.url,
                    rec.title,
                    rec.relevance_score,
                    rec.category,
                    json.dumps(rec.keywords),
                    rec.cluster_id,
                    int(rec.is_duplicate),
                    rec.query_source,
                    rec.canonical_url,
                    run_id,
                    user_query,
                    now,
                    rec.tavily_score,
                    now,
                )
            )
            self.conn.commit()
        except Exception as e:
            print(f'   [archive error] {e}')

    def get_recent_reflections(self, n=5):
        rows = self.conn.execute(
            'SELECT agent,issue,fix FROM reflection_log ORDER BY id DESC LIMIT ?', (n,)
        ).fetchall()
        return [{'agent': r[0], 'issue': r[1], 'fix': r[2]} for r in rows]

    def get_latest_run_id(self):
        row = self.conn.execute(
            "SELECT run_id FROM archived_urls WHERE run_id IS NOT NULL AND run_id != '' ORDER BY id DESC LIMIT 1"
        ).fetchone()
        return row[0] if row else None

    def get_archived_urls(self, run_id=None):
        use_run_id = run_id if run_id is not None else self.get_latest_run_id()

        if use_run_id:
            rows = self.conn.execute(
                'SELECT url,title,score,category,keywords,cluster_id,is_duplicate,query_source,canonical_url,run_id,user_query,run_at '
                'FROM archived_urls WHERE run_id = ? ORDER BY score DESC, id DESC',
                (use_run_id,)
            ).fetchall()
        else:
            rows = self.conn.execute(
                'SELECT url,title,score,category,keywords,cluster_id,is_duplicate,query_source,canonical_url,run_id,user_query,run_at '
                'FROM archived_urls ORDER BY score DESC, id DESC'
            ).fetchall()

        out = []
        for r in rows:
            try:
                kws = json.loads(r[4]) if r[4] else []
            except Exception:
                kws = []
            out.append({
                'url': r[0],
                'title': r[1],
                'score': r[2],
                'category': r[3],
                'keywords': kws,
                'cluster_id': r[5],
                'is_duplicate': bool(r[6]),
                'query_source': r[7],
                'canonical_url': r[8],
                'run_id': r[9],
                'user_query': r[10],
                'run_at': r[11],
            })
        return out

memory = MemoryAgent()
print('Memory agent (SQLite) initialised.')

Memory agent (SQLite) initialised.


## Cell 6 — Supervisor agent

In [7]:
class SupervisorAgent:
    def __init__(self, memory):
        self.memory = memory

    def _call_llm(self, prompt, max_tokens=512):
        try:
            out = replicate.run(
                REPLICATE_MODEL,
                input={
                    'prompt': prompt,
                    'max_tokens': max_tokens,
                    'temperature': 0.2,
                    'system_prompt': 'You are a precise JSON-only responder. No markdown.'
                }
            )
            return ''.join(out)
        except Exception as e:
            print(f'   [supervisor llm error] {e}')
            return '{}'

    def plan(self, n_urls, user_query, past_failures=None):
        ctx = ''
        if past_failures:
            ctx = 'Past failures to avoid: ' + json.dumps(past_failures[:3])

        prompt = (
            'You are a task planner for a URL validation pipeline.\n'
            + f'Topic: {user_query}\n'
            + 'URLs to process: ' + str(n_urls) + '\n'
            + ctx + '\n'
            + 'Reply with ONLY this JSON (no explanation):\n'
            + '{"plan_type": "sequential", "steps": ["fetch", "validate", "deduplicate"], '
            + '"rationale": "one sentence"}'
        )
        raw = self._call_llm(prompt)
        try:
            m = re.search(r'\{.*\}', raw, re.DOTALL)
            plan = json.loads(m.group()) if m else {}
        except Exception:
            plan = {'plan_type': 'sequential', 'steps': ['fetch', 'validate', 'deduplicate'], 'rationale': 'fallback'}

        plan_type = 'distributed' if n_urls > 1000 else plan.get('plan_type', 'sequential')
        self.memory.log_routing('supervisor', f'plan={plan_type}', plan.get('rationale', ''))
        print(f'   [supervisor] Plan: {plan_type} | {plan.get("rationale", "")}')
        return plan

    def replan(self, reason):
        print(f'   [supervisor] Replanning: {reason}')
        self.memory.log_routing('supervisor', 'replan', reason)


class QueryPlannerAgent:
    def __init__(self, memory):
        self.memory = memory

    def _call_llm(self, prompt):
        try:
            out = replicate.run(
                REPLICATE_MODEL,
                input={
                    'prompt': prompt,
                    'max_tokens': 700,
                    'temperature': 0.1,
                    'system_prompt': 'Return only valid JSON.'
                }
            )
            return ''.join(out)
        except Exception as e:
            self.memory.log_routing('query_planner', 'llm_error', str(e)[:160])
            return '{}'

    def _normalize_tokens(self, text):
        tokens = re.findall(r'[a-z0-9][a-z0-9-]+', (text or '').lower())
        stop = {
            'best', 'top', 'list', 'guide', 'latest', 'news', 'report', 'reviews', 'review',
            'official', 'website', 'websites', 'info', 'information', 'about', 'for', 'the',
            'and', 'with', 'from', 'what', 'who', 'how', 'why'
        }
        return [t for t in tokens if len(t) > 2 and t not in stop]

    def _extract_anchor_terms(self, user_query):
        q = (user_query or '').strip().lower()
        terms = set(self._normalize_tokens(q))

        # Keep author/director style anchors strongly tied to the intent
        m = re.search(r'(.+?)\s+by\s+(.+)', q)
        if m:
            left = self._normalize_tokens(m.group(1))
            right = self._normalize_tokens(m.group(2))
            terms.update(left)
            terms.update(right)

        # fallback to raw tokens if query is very short
        if not terms:
            terms = set(re.findall(r'[a-z0-9]+', q))
        return terms

    def _min_anchor_hits(self, anchor_terms):
        # Require stronger overlap for richer queries to prevent semantic drift
        if len(anchor_terms) >= 3:
            return 2
        return 1

    def _is_on_topic(self, candidate_query, anchor_terms):
        cand = set(self._normalize_tokens(candidate_query))
        if not cand:
            return False
        hits = len(cand & anchor_terms)
        return hits >= self._min_anchor_hits(anchor_terms)

    def _fallback_queries(self, user_query, n=MAX_QUERY_VARIANTS):
        # Every fallback query includes the full user query to keep strict intent alignment
        q = user_query.strip()
        templates = [
            q,
            f'"{q}"',
            f'{q} plot summary',
            f'{q} cast and crew',
            f'{q} reviews',
            f'{q} release details',
            f'{q} explained',
            f'{q} official sources',
            f'{q} analysis',
            f'{q} interviews',
            f'{q} critical reception',
            f'{q} facts',
        ]
        out, seen = [], set()
        for item in templates:
            v = item.strip()
            key = v.lower()
            if not v or key in seen:
                continue
            seen.add(key)
            out.append(v)
            if len(out) >= n:
                break
        return out

    def _dedupe_and_filter(self, queries, user_query, n):
        anchor_terms = self._extract_anchor_terms(user_query)
        clean, seen = [], set()
        dropped = 0

        for q in queries:
            if not isinstance(q, str):
                dropped += 1
                continue
            candidate = re.sub(r'\s+', ' ', q.strip())
            if not candidate:
                dropped += 1
                continue

            key = candidate.lower()
            if key in seen:
                continue

            if not self._is_on_topic(candidate, anchor_terms):
                dropped += 1
                continue

            seen.add(key)
            clean.append(candidate)
            if len(clean) >= n:
                break

        return clean, dropped

    def generate_queries(self, user_query, n=MAX_QUERY_VARIANTS):
        prompt = (
            'Generate strict web search queries for a single user intent.\n'
            + f'User topic: {user_query}\n'
            + f'Return between 8 and {n} unique queries.\n'
            + 'Rules: each query must stay anchored to the same entities/title in the user topic. '
            + 'Do NOT broaden to generic themes or reinterpret intent.\n'
            + 'Return ONLY JSON: {"queries": ["q1", "q2", ...]}'
        )
        raw = self._call_llm(prompt)

        parsed = []
        try:
            m = re.search(r'\{.*\}', raw, re.DOTALL)
            data = json.loads(m.group()) if m else {}
            parsed = data.get('queries', []) if isinstance(data, dict) else []
        except Exception:
            parsed = []

        clean, dropped = self._dedupe_and_filter(parsed, user_query, n)

        if len(clean) < 8:
            fallback = self._fallback_queries(user_query, n=n)
            fallback_clean, fb_dropped = self._dedupe_and_filter(fallback, user_query, n)
            clean = fallback_clean
            dropped += fb_dropped
            self.memory.log_routing('query_planner', 'fallback_used', f'query={user_query[:80]}')

        clean = clean[:n]
        self.memory.log_routing(
            'query_planner',
            f'generated={len(clean)} dropped_offtopic={dropped}',
            user_query[:120]
        )
        print(f'   [query planner] Generated {len(clean)} strict queries | Dropped off-topic: {dropped}')
        return clean


supervisor = SupervisorAgent(memory)
query_planner = QueryPlannerAgent(memory)
print('Supervisor + QueryPlanner agents ready.')

Supervisor + QueryPlanner agents ready.


## Cell 7 — Tavily fetch agent

In [8]:
class TavilyFetchAgent:
    """
    Fetches URLs via Tavily across generated overlapping queries.
    Strategy: same URL returned by different queries = duplicate candidate.
    """

    def __init__(self, memory):
        self.client = TavilyClient(api_key=os.environ['TAVILY_API_KEY'])
        self.memory = memory

    def fetch(self, user_query, queries=None):
        from collections import Counter

        queries = queries or query_planner.generate_queries(user_query)
        records = []
        seen_url_query = set()

        for q in queries:
            print(f"   [tavily] '{q}'")
            try:
                resp = self.client.search(
                    query=q,
                    search_depth='advanced',
                    max_results=TAVILY_MAX_RESULTS,
                    include_raw_content=True
                )
                added = 0
                for r in resp.get('results', []):
                    url = r.get('url', '').strip()
                    if not url:
                        continue
                    if (url, q) in seen_url_query:
                        continue
                    seen_url_query.add((url, q))

                    content = r.get('raw_content') or r.get('content', '')
                    records.append(URLRecord(
                        url=url,
                        title=r.get('title', ''),
                        content=content,
                        tavily_score=round(r.get('score', 0.0), 4),
                        query_source=q,
                    ))
                    added += 1
                print(f'      -> {added} records | running total: {len(records)}')
            except Exception as e:
                print(f'      [error] {e}')
                self.memory.log_routing('tavily', f'query_failed: {q[:80]}', str(e)[:180])

        url_counts = Counter(r.url for r in records)
        unique_urls = len(url_counts)
        multi_query_urls = {u: c for u, c in url_counts.items() if c > 1}

        self.memory.log_routing(
            'tavily',
            f'user_query={user_query[:80]} | {len(records)} records | {unique_urls} unique | {len(multi_query_urls)} multi-query'
        )

        print(f"\n   [tavily] User query            : {user_query}")
        print(f'   [tavily] Raw records collected : {len(records)}')
        print(f'   [tavily] Unique URLs           : {unique_urls}')
        print(f'   [tavily] Multi-query URLs      : {len(multi_query_urls)}')
        if multi_query_urls:
            print('   [tavily] Top dupe candidates:')
            for u, c in sorted(multi_query_urls.items(), key=lambda x: -x[1])[:8]:
                print(f'      {c}x  {u[:90]}')
        return records


tavily_agent = TavilyFetchAgent(memory)
print('Tavily fetch agent ready.')

Tavily fetch agent ready.


## Cell 8 — Content agent

In [9]:
class ContentAgent:
    def __init__(self, memory):
        self.memory = memory
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (compatible; URLPipelineBot/1.0; +https://example.org/bot)'
        })

    def _extract_metadata(self, html, base_url=''):
        from urllib.parse import urljoin

        try:
            soup = BeautifulSoup(html, 'lxml')
        except Exception:
            soup = BeautifulSoup(html, 'html.parser')

        def get_meta(*keys):
            for key in keys:
                tag = soup.find('meta', attrs={'name': key}) or soup.find('meta', attrs={'property': key})
                if tag and tag.get('content'):
                    return tag.get('content').strip()
            return ''

        page_title = ''
        if soup.title and soup.title.string:
            page_title = soup.title.string.strip()

        canonical_url = ''
        for link in soup.find_all('link'):
            rel = link.get('rel') or []
            rel_text = ' '.join(rel).lower() if isinstance(rel, list) else str(rel).lower()
            href = (link.get('href') or '').strip()
            if 'canonical' in rel_text and href:
                canonical_url = urljoin(base_url, href)
                break

        html_tag = soup.find('html')
        lang = html_tag.get('lang', '').strip().lower() if html_tag else ''

        desc = get_meta('description', 'og:description', 'twitter:description')
        keyword_raw = get_meta('keywords')
        meta_keywords = [k.strip() for k in keyword_raw.split(',') if k.strip()][:12]

        site_name = get_meta('og:site_name', 'twitter:site')

        return {
            'title': page_title,
            'description': desc,
            'keywords': meta_keywords,
            'site_name': site_name,
            'canonical_url': canonical_url,
            'lang': lang,
        }

    def _scrape_metadata(self, url):
        try:
            resp = self.session.get(url, timeout=10, allow_redirects=True)
            ctype = resp.headers.get('content-type', '').lower()
            if ('text/html' not in ctype) and ('application/xhtml+xml' not in ctype):
                return {}
            return self._extract_metadata(resp.text, resp.url or url)
        except Exception as e:
            self.memory.log_routing('content', f'meta_fail: {url[:60]}', str(e)[:120])
            return {}

    def _is_blocked_page(self, rec):
        check = ' '.join([rec.page_title or '', rec.meta_description or '']).lower()
        blocked_markers = [
            'access denied', 'just a moment', 'attention required', 'captcha',
            'forbidden', 'blocked', 'temporarily unavailable', 'page not found', '404'
        ]
        return any(m in check for m in blocked_markers)

    def process(self, records):
        passed, skipped, meta_enriched = [], 0, 0
        for rec in records:
            meta = self._scrape_metadata(rec.url)

            rec.page_title       = meta.get('title', '') or rec.title
            rec.meta_description = meta.get('description', '')
            rec.meta_keywords    = meta.get('keywords', [])
            rec.site_name        = meta.get('site_name', '')
            rec.canonical_url    = meta.get('canonical_url', '')
            if meta.get('lang'):
                rec.language = meta.get('lang')
            if rec.page_title:
                rec.title = rec.page_title

            metadata_text = ' '.join([
                rec.page_title,
                rec.meta_description,
                ' '.join(rec.meta_keywords),
                rec.site_name
            ]).strip()

            text = (rec.content or '').strip()
            wc = len(text.split())
            meta_wc = len(metadata_text.split())

            if self._is_blocked_page(rec) and wc < 120:
                skipped += 1
                self.memory.log_routing('content', f'skip_blocked: {rec.url[:60]}', (rec.page_title or '')[:100])
                continue

            # Keep records with rich metadata even when raw body is short.
            if wc < MIN_WORD_COUNT and meta_wc >= 8:
                text = (text + '\n\n[METADATA]\n' + metadata_text).strip()
                wc = len(text.split())
                meta_enriched += 1

            if wc < MIN_WORD_COUNT and meta_wc < 8:
                skipped += 1
                self.memory.log_routing('content', f'skip: {rec.url[:60]}', f'word_count={wc}; meta_words={meta_wc}')
                continue

            rec.content = text
            hash_input = text + '|' + rec.meta_description + '|' + ','.join(rec.meta_keywords)
            rec.content_hash = hashlib.md5(hash_input.encode()).hexdigest()
            rec.word_count = wc
            passed.append(rec)

        print(f'   [content] Passed: {len(passed)} | Skipped: {skipped} | Metadata-enriched: {meta_enriched}')
        return passed

content_agent = ContentAgent(memory)
print('Content agent ready.')

Content agent ready.


## Cell 9 — Relevance agent

In [10]:
class RelevanceAgent:
    def __init__(self, memory, user_query, user_region=''):
        self.memory = memory
        self.user_query = user_query
        self.user_region = (user_region or '').strip()
        self._system = 'Respond only with valid JSON, no markdown, no explanation.'
        self._extra = ''

    def _normalize_category(self, raw):
        text = (raw or '').strip().lower()
        text = re.sub(r'[^a-z0-9 _\-/+]+', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip(' _-/+')
        if not text:
            return 'other'
        # keep short stable category labels
        return text[:40]

    def _build_prompt(self, rec):
        metadata_payload = {
            'url': rec.url,
            'page_title': rec.page_title or rec.title,
            'meta_description': rec.meta_description,
            'meta_keywords': rec.meta_keywords[:12],
            'site_name': rec.site_name,
            'canonical_url': rec.canonical_url,
            'query_source': rec.query_source,
        }
        metadata_json = json.dumps(metadata_payload, ensure_ascii=False)
        region_line = f"Target region (optional): {self.user_region}" if self.user_region else 'Target region (optional): <none>'

        return (
            "You are a relevance classifier for arbitrary user queries.\n"
            + f"User query: {self.user_query}\n"
            + region_line + '\n'
            + self._extra
            + '\nUse both scraped metadata and content.\n'
            + 'Respond ONLY with this exact JSON structure:\n'
            + '{"score": <float 0.0-1.0>, '
            + '"reasoning": "<one sentence>", '
            + '"keywords": ["kw1", "kw2", "kw3"], '
            + '"query_fit": <true or false>, '
            + '"region_match": <true or false>, '
            + '"category": "<short free-text label>"}\n\n'
            + 'Scraped metadata:\n' + metadata_json + '\n\n'
            + 'Content (first 2000 chars):\n' + rec.content[:2000]
        )

    def _call_llm(self, rec):
        prompt = self._build_prompt(rec)
        for attempt in range(2):
            try:
                out = replicate.run(
                    REPLICATE_MODEL,
                    input={
                        'prompt': prompt,
                        'max_tokens': 350,
                        'temperature': 0.1,
                        'system_prompt': self._system
                    }
                )
                raw = ''.join(out)
                m = re.search(r'\{[^{}]*\}', raw, re.DOTALL)
                if m:
                    return json.loads(m.group())
                print(f'      [relevance] no JSON found, raw: {raw[:120]}')
            except Exception as e:
                if attempt == 0:
                    time.sleep(2)
                else:
                    print(f'      [llm error] {e}')
        return {
            'score': 0.0,
            'reasoning': 'parse_failed',
            'keywords': [],
            'query_fit': False,
            'region_match': True,
            'category': 'other',
        }

    def classify(self, records):
        relevant, discarded = [], 0
        for i, rec in enumerate(records):
            print(f'   [relevance] {i+1}/{len(records)} {rec.url[:60]}')
            r = self._call_llm(rec)

            rec.relevance_score = round(float(r.get('score', 0.0)), 3)
            rec.reasoning       = r.get('reasoning', '')
            rec.keywords        = r.get('keywords', []) or rec.meta_keywords[:3]
            rec.query_fit       = bool(r.get('query_fit', False))
            rec.region_match    = bool(r.get('region_match', True)) if self.user_region else True
            rec.category        = self._normalize_category(r.get('category', 'other'))

            rec.is_relevant = (
                rec.relevance_score >= RELEVANCE_THRESHOLD
                and rec.query_fit
                and rec.region_match
            )

            if rec.is_relevant:
                relevant.append(rec)
                print(f'      PASS score={rec.relevance_score} cat={rec.category}')
            else:
                discarded += 1
                print(
                    f'      FAIL score={rec.relevance_score} '
                    f'query_fit={rec.query_fit} region_match={rec.region_match}'
                )
                self.memory.log_routing(
                    'relevance',
                    f'discard: {rec.url[:60]}',
                    rec.reasoning[:120],
                )

        print(f'   [relevance] Relevant: {len(relevant)} | Discarded: {discarded}')
        return relevant

    def update_prompt(self, recommendation):
        old_hash = hashlib.md5(self._extra.encode()).hexdigest()[:8]
        self._extra += f'\n# Improvement: {recommendation}'
        new_hash = hashlib.md5(self._extra.encode()).hexdigest()[:8]
        self.memory.log_reflection(
            'relevance',
            f'Prompt updated: {recommendation[:80]}',
            f'old={old_hash} new={new_hash}'
        )


relevance_agent = RelevanceAgent(memory, USER_QUERY, USER_REGION)
print('Relevance agent ready.')

Relevance agent ready.


## Cell 10 — Dupe agent

In [11]:
class DupeAgent:
    def __init__(self, memory, user_query=''):
        self.memory = memory
        self.user_query = user_query or ''
        self.query_tokens = self._extract_query_tokens(self.user_query)
        self.last_metrics = {
            'total_records': 0,
            'clustered_count': 0,
            'singleton_count': 0,
            'dupe_count': 0,
        }
        print('   [dupe] Loading sentence-transformer...')
        self.model = SentenceTransformer('all-MiniLM-L6-v2')

    def _extract_query_tokens(self, query):
        toks = re.findall(r'[a-z0-9][a-z0-9-]+', (query or '').lower())
        generic = {
            'best', 'top', 'list', 'guide', 'latest', 'news', 'report', 'companies',
            'products', 'services', 'website', 'official'
        }
        return {t for t in toks if len(t) > 2 and t not in generic}

    def _normalize_url(self, url):
        from urllib.parse import urlparse, parse_qsl

        try:
            p = urlparse((url or '').strip())
            host = (p.netloc or '').lower()
            if host.startswith('www.'):
                host = host[4:]

            path = (p.path or '/').rstrip('/').lower()
            path = re.sub(r'/+', '/', path)

            drop_keys = {'ref', 'source', 'fbclid', 'gclid'}
            kept = []
            for k, v in parse_qsl(p.query, keep_blank_values=False):
                lk = k.lower()
                if lk.startswith('utm_') or lk in drop_keys:
                    continue
                kept.append((k, v))
            kept.sort()
            q = '&'.join([f"{k}={v}" for k, v in kept])
            return f"{host}{path}" + (f"?{q}" if q else '')
        except Exception:
            return (url or '').strip().lower().rstrip('/')

    def _url_host_path(self, url):
        from urllib.parse import urlparse

        try:
            p = urlparse((url or '').strip())
            host = (p.netloc or '').lower()
            if host.startswith('www.'):
                host = host[4:]
            path = (p.path or '/').rstrip('/').lower()
            path = re.sub(r'/+', '/', path)
            return host, path
        except Exception:
            return '', ''

    def _token_set(self, rec):
        text = ' '.join([
            rec.title or '',
            rec.page_title or '',
            rec.meta_description or '',
            ' '.join(rec.keywords or []),
            ' '.join(rec.meta_keywords or []),
            rec.site_name or ''
        ]).lower()

        aliases = {
            'start up': 'startup',
            'open source': 'opensource',
        }
        for k, v in aliases.items():
            text = text.replace(k, v)

        tokens = re.findall(r'[a-z0-9][a-z0-9-]+', text)
        stop = {
            'the', 'and', 'for', 'with', 'this', 'that', 'from', 'are', 'was', 'www', 'http', 'https',
            'company', 'companies', 'platform', 'solution', 'solutions', 'market', 'overview',
            '2024', '2025', '2026', 'official', 'home', 'page', 'com', 'org', 'net'
        }
        stop = stop | self.query_tokens
        return {t for t in tokens if len(t) > 2 and t not in stop}

    def _pair_features(self, i, j, records, embeddings, token_sets):
        import difflib

        cosine = float(np.dot(embeddings[i], embeddings[j]))
        overlap = len(token_sets[i] & token_sets[j])
        same_category = records[i].category == records[j].category and records[i].category != 'other'

        host_i, path_i = self._url_host_path(records[i].canonical_url or records[i].url)
        host_j, path_j = self._url_host_path(records[j].canonical_url or records[j].url)
        same_host = bool(host_i and host_j and host_i == host_j)
        path_sim = difflib.SequenceMatcher(None, path_i, path_j).ratio() if (path_i and path_j) else 0.0

        mk_i = {k.lower().strip() for k in (records[i].meta_keywords or []) if k.strip()}
        mk_j = {k.lower().strip() for k in (records[j].meta_keywords or []) if k.strip()}
        meta_overlap = len(mk_i & mk_j)

        return {
            'cosine': cosine,
            'overlap': overlap,
            'same_category': same_category,
            'same_host': same_host,
            'path_sim': path_sim,
            'meta_overlap': meta_overlap,
        }

    def _is_true_duplicate(self, feats):
        if feats['same_host'] and feats['path_sim'] >= 0.96:
            return True
        if feats['cosine'] >= 0.90 and (feats['same_host'] or feats['overlap'] >= 4 or feats['meta_overlap'] >= 3):
            return True
        return False

    def _add_to_cluster(self, cluster_map, cid, idx):
        cluster_map.setdefault(cid, [])
        if idx not in cluster_map[cid]:
            cluster_map[cid].append(idx)

    def _auto_attach_dupes(self, records, embeddings, labels, cluster_map, token_sets, next_cluster):
        by_norm = {}
        for i, r in enumerate(records):
            key = self._normalize_url(r.canonical_url or r.url)
            by_norm.setdefault(key, []).append(i)

        for idxs in by_norm.values():
            if len(idxs) < 2:
                continue
            existing = [labels[i] for i in idxs if labels[i] != -1]
            cid = existing[0] if existing else next_cluster
            if not existing:
                next_cluster += 1
            for idx in idxs:
                labels[idx] = cid
                self._add_to_cluster(cluster_map, cid, idx)

        reps = {}
        for cid, idxs in cluster_map.items():
            if idxs:
                reps[cid] = max(idxs, key=lambda k: (records[k].word_count, records[k].relevance_score))

        for i in range(len(records)):
            if labels[i] != -1:
                continue
            best_cid = -1
            best_cos = -1.0
            for cid, rep in reps.items():
                feats = self._pair_features(i, rep, records, embeddings, token_sets)
                if self._is_true_duplicate(feats) and feats['cosine'] > best_cos:
                    best_cos = feats['cosine']
                    best_cid = cid
            if best_cid != -1:
                labels[i] = best_cid
                self._add_to_cluster(cluster_map, best_cid, i)

        return next_cluster

    def cluster(self, records):
        if len(records) < 2:
            if len(records) == 1:
                records[0].cluster_id = 0
            self.last_metrics = {
                'total_records': len(records),
                'clustered_count': len(records),
                'singleton_count': len(records),
                'dupe_count': 0,
            }
            return records

        texts = [
            f"{r.title} {r.page_title} {r.meta_description} {' '.join(r.keywords)} {' '.join(r.meta_keywords)} {r.site_name} {r.category} {r.content[:500]}"
            for r in records
        ]
        print(f'   [dupe] Embedding {len(texts)} URLs (content + metadata)...')
        embeddings = self.model.encode(texts, show_progress_bar=True, normalize_embeddings=True)

        for i, rec in enumerate(records):
            rec.embedding = embeddings[i].tolist()

        labels = [-1] * len(records)
        cluster_map = {}
        next_cluster = 0

        # Stage 1: exact canonical URL duplicates
        by_norm_url = {}
        for i, rec in enumerate(records):
            key = self._normalize_url(rec.canonical_url or rec.url)
            by_norm_url.setdefault(key, []).append(i)

        for idxs in by_norm_url.values():
            if len(idxs) < 2:
                continue
            cid = next_cluster
            next_cluster += 1
            cluster_map[cid] = idxs[:]
            for idx in idxs:
                labels[idx] = cid

        # Stage 2: greedy clustering
        token_sets = [self._token_set(r) for r in records]
        order = sorted(
            [i for i in range(len(records)) if labels[i] == -1],
            key=lambda i: (records[i].relevance_score, records[i].word_count),
            reverse=True,
        )

        provisional = []
        for i in order:
            best_k = -1
            best_score = -1.0
            emb_i = embeddings[i]

            for k, c in enumerate(provisional):
                token_overlap = len(token_sets[i] & c['token_union'])
                if token_overlap == 0 and records[i].category != c['category']:
                    continue

                centroid_cos = float(np.dot(emb_i, c['centroid']))
                anchor = c['members'][0]
                feats = self._pair_features(i, anchor, records, embeddings, token_sets)

                match = False
                if feats['same_host'] and feats['path_sim'] >= 0.88:
                    match = True
                elif records[i].category == c['category'] and records[i].category != 'other' and centroid_cos >= 0.72 and token_overlap >= 2:
                    match = True
                elif centroid_cos >= 0.79 and token_overlap >= 3:
                    match = True

                if match:
                    score = centroid_cos + (0.03 * min(token_overlap, 4))
                    if score > best_score:
                        best_score = score
                        best_k = k

            if best_k == -1:
                provisional.append({
                    'members': [i],
                    'sum_vec': emb_i.copy(),
                    'count': 1,
                    'centroid': emb_i,
                    'token_union': set(token_sets[i]),
                    'category': records[i].category,
                })
            else:
                c = provisional[best_k]
                c['members'].append(i)
                c['token_union'].update(token_sets[i])
                c['sum_vec'] = c['sum_vec'] + emb_i
                c['count'] += 1
                centroid = c['sum_vec'] / c['count']
                norm = np.linalg.norm(centroid)
                c['centroid'] = centroid / norm if norm > 0 else centroid

        for c in provisional:
            if len(c['members']) < 2:
                continue
            cid = next_cluster
            next_cluster += 1
            cluster_map[cid] = c['members']
            for idx in c['members']:
                labels[idx] = cid

        # Stage 3: auto-attach strict duplicates
        next_cluster = self._auto_attach_dupes(records, embeddings, labels, cluster_map, token_sets, next_cluster)

        # Stage 4: merge near-topic leftovers to existing clusters
        merged_into_existing = 0
        remaining = [i for i in range(len(records)) if labels[i] == -1]

        if remaining and cluster_map:
            profiles = []
            for cid, idxs in cluster_map.items():
                if not idxs:
                    continue
                vecs = embeddings[idxs]
                centroid = np.mean(vecs, axis=0)
                norm = np.linalg.norm(centroid)
                centroid = centroid / norm if norm > 0 else centroid

                token_union = set()
                cat_counts = {}
                for idx in idxs:
                    token_union.update(token_sets[idx])
                    cat = records[idx].category or 'other'
                    cat_counts[cat] = cat_counts.get(cat, 0) + 1
                category = max(cat_counts, key=cat_counts.get)
                rep = max(idxs, key=lambda k: (records[k].word_count, records[k].relevance_score))
                profiles.append((cid, centroid, token_union, category, rep))

            for i in remaining:
                best_cid = -1
                best_score = -1.0
                for cid, centroid, token_union, category, rep in profiles:
                    token_overlap = len(token_sets[i] & token_union)
                    if token_overlap == 0 and records[i].category != category:
                        continue

                    centroid_cos = float(np.dot(embeddings[i], centroid))
                    feats = self._pair_features(i, rep, records, embeddings, token_sets)

                    match = False
                    if feats['same_host'] and feats['path_sim'] >= 0.84:
                        match = True
                    elif records[i].category == category and records[i].category != 'other' and centroid_cos >= 0.68 and token_overlap >= 1:
                        match = True
                    elif centroid_cos >= 0.75 and token_overlap >= 2:
                        match = True
                    elif feats['cosine'] >= 0.82 and (feats['overlap'] >= 2 or feats['meta_overlap'] >= 1):
                        match = True

                    if match:
                        score = centroid_cos + (0.03 * min(token_overlap, 4)) + (0.02 if records[i].category == category else 0.0)
                        if score > best_score:
                            best_score = score
                            best_cid = cid

                if best_cid != -1:
                    labels[i] = best_cid
                    self._add_to_cluster(cluster_map, best_cid, i)
                    merged_into_existing += 1

        # Stage 5: pair remaining leftovers
        paired_leftovers = 0
        remaining = [i for i in range(len(records)) if labels[i] == -1]
        used = set()

        for pos, i in enumerate(remaining):
            if i in used:
                continue

            best_j = -1
            best_score = -1.0
            for j in remaining[pos + 1:]:
                if j in used:
                    continue

                if (records[i].category != 'other' and records[j].category != 'other'
                        and records[i].category != records[j].category):
                    continue

                token_overlap = len(token_sets[i] & token_sets[j])
                if token_overlap == 0:
                    continue

                feats = self._pair_features(i, j, records, embeddings, token_sets)

                match = False
                if feats['same_host'] and feats['path_sim'] >= 0.82:
                    match = True
                elif records[i].category == records[j].category and records[i].category != 'other' and feats['cosine'] >= 0.70 and token_overlap >= 2:
                    match = True
                elif feats['cosine'] >= 0.76 and (feats['overlap'] >= 2 or feats['meta_overlap'] >= 1):
                    match = True

                if match:
                    score = feats['cosine'] + (0.03 * min(token_overlap, 4))
                    if score > best_score:
                        best_score = score
                        best_j = j

            if best_j != -1:
                cid = next_cluster
                next_cluster += 1
                cluster_map[cid] = [i, best_j]
                labels[i] = cid
                labels[best_j] = cid
                used.add(i)
                used.add(best_j)
                paired_leftovers += 2

        # Stage 6: assign singleton cluster IDs for all still-unclustered records
        singleton_count = 0
        for i in range(len(records)):
            if labels[i] != -1:
                continue
            labels[i] = next_cluster
            cluster_map[next_cluster] = [i]
            next_cluster += 1
            singleton_count += 1

        for i, rec in enumerate(records):
            rec.cluster_id = int(labels[i])

        dupe_count = 0
        for label, idxs in cluster_map.items():
            if len(idxs) < 2:
                continue
            canonical = max(idxs, key=lambda k: (records[k].word_count, records[k].relevance_score))
            canon_norm = self._normalize_url(records[canonical].canonical_url or records[canonical].url)

            for idx in idxs:
                if idx == canonical:
                    continue
                idx_norm = self._normalize_url(records[idx].canonical_url or records[idx].url)
                feats = self._pair_features(idx, canonical, records, embeddings, token_sets)
                if idx_norm == canon_norm or self._is_true_duplicate(feats):
                    records[idx].is_duplicate = True
                    dupe_count += 1
                    self.memory.log_routing(
                        'dupe',
                        f'dupe: {records[idx].url[:55]}',
                        f'cluster={label} canonical={records[canonical].url[:40]}'
                    )

        clustered_count = sum(1 for r in records if r.cluster_id >= 0)
        canonical_recs = [r for r in records if not r.is_duplicate]
        self.last_metrics = {
            'total_records': len(records),
            'clustered_count': clustered_count,
            'singleton_count': singleton_count,
            'dupe_count': dupe_count,
            'merged_into_existing': merged_into_existing,
            'paired_leftovers': paired_leftovers,
            'canonical_candidates': len(canonical_recs),
            'total_clusters': len(cluster_map),
        }

        print(
            f"   [dupe] Clusters: {len(cluster_map)} | "
            f"Merged to existing: {merged_into_existing} | "
            f"Paired leftovers: {paired_leftovers} | "
            f"Singleton clusters: {singleton_count} | "
            f"Dupes: {dupe_count} | "
            f"Canonical candidates: {len(canonical_recs)}"
        )

        return records


dupe_agent = DupeAgent(memory, USER_QUERY)
print('Dupe agent ready.')

   [dupe] Loading sentence-transformer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Dupe agent ready.


## Cell 11 — Eval agent

In [12]:
class EvalAgent:
    def __init__(self, memory, relevance_agent, user_query=''):
        self.memory = memory
        self.relevance_agent = relevance_agent
        self.user_query = user_query or 'user query'

    def _call_llm(self, summary):
        prompt = (
            'You are a QA reviewer for a URL validation pipeline.\n'
            + f"Topic: '{self.user_query}'\n"
            + 'Review the results below and reply with ONLY this JSON:\n'
            + '{"accuracy_estimate": <0.0-1.0>, '
            + '"false_positives": ["url"], '
            + '"missed_duplicates": ["url"], '
            + '"issues": ["issue description"], '
            + '"recommendations": ["recommendation"]}\n\n'
            + 'Results:\n'
            + summary[:3000]
        )
        for attempt in range(2):
            try:
                out = replicate.run(
                    REPLICATE_MODEL,
                    input={
                        'prompt': prompt,
                        'max_tokens': 600,
                        'temperature': 0.2,
                        'system_prompt': 'Respond only with valid JSON.'
                    }
                )
                raw = ''.join(out)
                m = re.search(r'\{.*\}', raw, re.DOTALL)
                if m:
                    return json.loads(m.group())
            except Exception as e:
                if attempt == 0:
                    time.sleep(1)
                else:
                    print(f'      [eval error] {e}')
        return {
            'accuracy_estimate': 0.95,
            'false_positives': [],
            'missed_duplicates': [],
            'issues': [],
            'recommendations': [],
        }

    def critique(self, records):
        summary = json.dumps([
            {
                'url': r.url,
                'score': r.relevance_score,
                'category': r.category,
                'query_fit': r.query_fit,
                'keywords': r.keywords[:3],
                'reasoning': r.reasoning,
            }
            for r in records
        ], indent=2)

        print(f'   [eval] Critiquing {len(records)} URLs...')
        result = self._call_llm(summary)
        acc = float(result.get('accuracy_estimate', 0.95))
        print(f'   [eval] accuracy_estimate = {acc:.2f}')

        for issue in result.get('issues', []):
            self.memory.log_reflection('eval', issue, 'logged')
        for rec in result.get('recommendations', []):
            self.relevance_agent.update_prompt(rec)

        if acc < ACCURACY_PAUSE:
            with open('human_review.csv', 'w') as f:
                f.write('url,score,category,reasoning\n')
                for r in records:
                    f.write(f'{r.url},{r.relevance_score},{r.category},{r.reasoning}\n')
            print(f'   [eval] PAUSED accuracy={acc:.2f} < {ACCURACY_PAUSE}. human_review.csv written.')
            result['action'] = 'pause'
        elif acc < ACCURACY_RERUN:
            print(f'   [eval] accuracy={acc:.2f} < {ACCURACY_RERUN} — flagging rerun')
            result['action'] = 'rerun'
        else:
            print(f'   [eval] Quality gate PASSED ({acc:.2f})')
            result['action'] = 'archive'

        result['accuracy_estimate'] = acc
        return result


eval_agent = EvalAgent(memory, relevance_agent, USER_QUERY)
print('Eval agent ready.')

Eval agent ready.


## Cell 12 — Archive agent

In [13]:
class ArchiveAgent:
    def __init__(self, memory):
        self.memory = memory
        self.index = None
        self.indexed_urls = []

    def store(self, records, run_id, user_query):
        stored, vecs = 0, []
        for rec in records:
            self.memory.archive_url(rec, run_id=run_id, user_query=user_query)
            if rec.embedding:
                vecs.append(np.array(rec.embedding, dtype='float32'))
                self.indexed_urls.append(rec.url)
            stored += 1

        if vecs:
            dim = len(vecs[0])
            self.index = faiss.IndexFlatL2(dim)
            self.index.add(np.stack(vecs))
            print(f'   [archive] FAISS index: {self.index.ntotal} vectors')
        print(f'   [archive] Stored {stored} URLs to SQLite.')
        return stored

    def search_similar(self, query_emb, top_k=5):
        if self.index is None or self.index.ntotal == 0:
            return []
        q = np.array([query_emb], dtype='float32')
        _, idx = self.index.search(q, min(top_k, self.index.ntotal))
        return [self.indexed_urls[i] for i in idx[0] if i < len(self.indexed_urls)]


archive_agent = ArchiveAgent(memory)
print('Archive agent ready.')

Archive agent ready.


## Cell 13 — LangGraph orchestration

In [14]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

class PipelineState(TypedDict):
    records:      list
    plan:         dict
    eval_result:  dict
    rerun_count:  int
    final_count:  int
    run_metrics:  dict


def node_supervisor(state):
    print('\n-- SUPERVISOR --')
    past = memory.get_recent_reflections(5)
    plan = supervisor.plan(len(state.get('records', [])), USER_QUERY, past)
    return {**state, 'plan': plan}


def node_tavily(state):
    print('\n-- TAVILY FETCH --')
    records = tavily_agent.fetch(USER_QUERY)
    if not records:
        supervisor.replan('Tavily returned 0 results')
    m = dict(state.get('run_metrics', {}))
    m['fetched_count'] = len(records)
    return {**state, 'records': records, 'run_metrics': m}


def node_content(state):
    print('\n-- CONTENT --')
    records = content_agent.process(state['records'])
    m = dict(state.get('run_metrics', {}))
    m['content_passed_count'] = len(records)
    return {**state, 'records': records, 'run_metrics': m}


def node_relevance(state):
    print('\n-- RELEVANCE --')
    records = relevance_agent.classify(state['records'])
    m = dict(state.get('run_metrics', {}))
    m['relevant_count'] = len(records)
    return {**state, 'records': records, 'run_metrics': m}


def node_dupe(state):
    print('\n-- DUPE DETECTION --')
    records = dupe_agent.cluster(state['records'])
    m = dict(state.get('run_metrics', {}))
    m.update(dupe_agent.last_metrics)
    return {**state, 'records': records, 'run_metrics': m}


def node_eval(state):
    print('\n-- EVAL --')
    return {**state, 'eval_result': eval_agent.critique(state['records'])}


def node_archive(state):
    print('\n-- ARCHIVE --')
    stored = archive_agent.store(state['records'], run_id=RUN_ID, user_query=USER_QUERY)
    m = dict(state.get('run_metrics', {}))
    m['archived_count'] = stored
    return {**state, 'final_count': stored, 'run_metrics': m}


def route_after_tavily(state):
    return 'content' if state.get('records') else 'supervisor'


def route_after_eval(state):
    action = state.get('eval_result', {}).get('action', 'archive')
    if action == 'pause':
        return END
    if action == 'rerun' and state.get('rerun_count', 0) < 1:
        return 'relevance'
    return 'archive'


builder = StateGraph(PipelineState)
for name, fn in [
    ('supervisor', node_supervisor),
    ('tavily', node_tavily),
    ('content', node_content),
    ('relevance', node_relevance),
    ('dupe', node_dupe),
    ('eval', node_eval),
    ('archive', node_archive),
]:
    builder.add_node(name, fn)

builder.set_entry_point('supervisor')
builder.add_edge('supervisor', 'tavily')
builder.add_conditional_edges('tavily', route_after_tavily, {'content': 'content', 'supervisor': 'supervisor'})
builder.add_edge('content', 'relevance')
builder.add_edge('relevance', 'dupe')
builder.add_edge('dupe', 'eval')
builder.add_conditional_edges('eval', route_after_eval, {'archive': 'archive', 'relevance': 'relevance', END: END})
builder.add_edge('archive', END)

pipeline = builder.compile()
print('LangGraph pipeline compiled and ready.')

LangGraph pipeline compiled and ready.


## Cell 14 — Run the pipeline

In [15]:
print('=' * 60)
print(f'  USER_QUERY : {USER_QUERY}')
print(f'  USER_REGION: {USER_REGION if USER_REGION else "<none>"}')
print(f'  RUN_ID     : {RUN_ID}')
print(f'  MODEL      : {REPLICATE_MODEL}')
print(f'  THRESHOLD  : {RELEVANCE_THRESHOLD}')
print('=' * 60)

final_state = pipeline.invoke({
    'records': [],
    'plan': {},
    'eval_result': {},
    'rerun_count': 0,
    'final_count': 0,
    'run_metrics': {},
})

print('\n' + '=' * 60)
print('  PIPELINE COMPLETE')
print(f"  Final archived URLs : {final_state.get('final_count', 0)}")
acc = final_state.get('eval_result', {}).get('accuracy_estimate', 'n/a')
print(f'  Eval accuracy       : {acc}')
print(f"  Run metrics         : {final_state.get('run_metrics', {})}")
print('=' * 60)

  USER_QUERY : INSOMNIA by Christopher Nolan
  USER_REGION: <none>
  RUN_ID     : 6bada1e3-6f47-4cf3-a1e1-378128941e56
  MODEL      : meta/meta-llama-3-8b-instruct
  THRESHOLD  : 0.7

-- SUPERVISOR --
   [supervisor] Plan: sequential | To ensure efficient processing of URLs, we will execute tasks in a sequential manner.

-- TAVILY FETCH --
   [query planner] Generated 9 strict queries | Dropped off-topic: 0
   [tavily] 'Christopher Nolan's film Insomnia'
      -> 15 records | running total: 15
   [tavily] 'Insomnia movie directed by Christopher Nolan'
      -> 15 records | running total: 30
   [tavily] 'Christopher Nolan's 2002 film Insomnia'
      -> 15 records | running total: 45
   [tavily] 'Insomnia film review by Christopher Nolan'
      -> 15 records | running total: 60
   [tavily] 'Christopher Nolan's Insomnia movie analysis'
      -> 15 records | running total: 75
   [tavily] 'Insomnia movie directed by Christopher Nolan cast'
      -> 12 records | running total: 87
   [tavily]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

   [dupe] Clusters: 20 | Merged to existing: 6 | Paired leftovers: 0 | Singleton clusters: 7 | Dupes: 35 | Canonical candidates: 32

-- EVAL --
   [eval] Critiquing 67 URLs...
   [eval] accuracy_estimate = 0.90
   [reflection] eval: Some results may be film analysis or reviews instead of the actual film
   [reflection] relevance: Prompt updated: Consider using more specific keywords or query phrases to filter
   [eval] Quality gate PASSED (0.90)

-- ARCHIVE --
   [archive] FAISS index: 67 vectors
   [archive] Stored 67 URLs to SQLite.

  PIPELINE COMPLETE
  Final archived URLs : 67
  Eval accuracy       : 0.9
  Run metrics         : {'fetched_count': 131, 'content_passed_count': 118, 'relevant_count': 67, 'total_records': 67, 'clustered_count': 67, 'singleton_count': 7, 'dupe_count': 35, 'merged_into_existing': 6, 'paired_leftovers': 0, 'canonical_candidates': 32, 'total_clusters': 20, 'archived_count': 67}


## Cell 15 — View results

In [16]:
import pandas as pd

archived = memory.get_archived_urls(run_id=RUN_ID)
if archived:
    df = pd.DataFrame(archived)
    df['score'] = df['score'].round(3)
    df['keywords'] = df['keywords'].apply(lambda k: ', '.join(k[:3]) if k else '')
    pd.set_option('display.max_colwidth', 80)

    print(f'Archived {len(df)} URLs for current run: {RUN_ID}\n')
    display(
        df[
            ['cluster_id', 'url', 'title', 'score', 'category', 'keywords', 'is_duplicate', 'query_source']
        ].sort_values(['cluster_id', 'score'], ascending=[True, False])
    )
else:
    print('No archived URLs yet for current run.')

Archived 67 URLs for current run: 6bada1e3-6f47-4cf3-a1e1-378128941e56



,cluster_id,url,title,score,category,keywords,is_duplicate,query_source
4,0,https://variety.com/2002/film/awards/insomnia-1117877182/,Insomnia,0.85,film,"Christopher Nolan, Insomnia, film",False,Christopher Nolan's Insomnia film awards and nominations
38,0,https://www.reddit.com/r/FIlm/comments/1gvvs4o/anyone_seen_christopher_nolan...,Anyone seen Christopher Nolan's Insomnia (2002)? : r/FIlm - Reddit,0.85,film,"Christopher Nolan, Insomnia, Al Pacino",False,Christopher Nolan's 2002 film Insomnia
42,0,https://www.reddit.com/r/FIlm/comments/1gvvs4o/anyone_seen_christopher_nolan...,Anyone seen Christopher Nolan's Insomnia (2002)? : r/FIlm,0.85,movie,"Insomnia, Christopher Nolan, Al Pacino",False,Insomnia movie directed by Christopher Nolan
51,0,https://www.reddit.com/r/FIlm/comments/1gvvs4o/anyone_seen_christopher_nolan...,Anyone seen Christopher Nolan's Insomnia (2002)? : r/FIlm,0.85,film,"Christopher Nolan, Insomnia, remake",False,Christopher Nolan's film Insomnia
6,1,https://en.wikipedia.org/wiki/Insomnia_(2002_film),Insomnia (2002 film) - Wikipedia,0.85,film,"Christopher Nolan, Insomnia, film",True,Christopher Nolan's Insomnia film awards and nominations
...,...,...,...,...,...,...,...,...
9,15,https://mubi.com/en/films/insomnia-2002/trailer,TRAILER - Insomnia (2002),0.85,thriller/mystery,"insomnia, christopher nolan, murder",False,Insomnia movie by Christopher Nolan trailer
66,16,https://en.wikipedia.org/wiki/List_of_awards_and_nominations_received_by_Chr...,List of awards and nominations received by Christopher Nolan - Wikipedia,0.70,film and entertainment,"Christopher Nolan, film director, Insomnia",False,Christopher Nolan's Insomnia film awards and nominations
3,17,https://www.facebook.com/cinemagicuniverse/posts/number-of-oscar-nominations...,Number of Oscar nominations for 🎥... - Cinemagic Universe,0.85,film industry,"Christopher Nolan, Oscar nominations, film awards",False,Christopher Nolan's Insomnia film awards and nominations
52,18,https://www.britannica.com/topic/Insomnia-film-by-Nolan,Just a moment...,0.80,film and entertainment,"Christopher Nolan, Insomnia, film",False,Christopher Nolan's Insomnia film awards and nominations


## Cell 16 — View agent logs & reflections

In [17]:
import pandas as pd

conn = sqlite3.connect(DB_PATH)
print('-- ROUTING LOG --')
display(pd.read_sql(
    'SELECT ts, agent, decision, reasoning FROM routing_log ORDER BY id DESC LIMIT 40', conn))
print('\n-- REFLECTION LOG --')
reflect = pd.read_sql(
    'SELECT ts, agent, issue, fix FROM reflection_log ORDER BY id DESC', conn)
if len(reflect):
    display(reflect)
else:
    print('No reflections logged (pipeline ran cleanly).')
conn.close()

-- ROUTING LOG --


,ts,agent,decision,reasoning
0,05:30:51,dupe,dupe: https://www.youtube.com/watch?v=PrtP4KuLvMQ,cluster=12 canonical=https://www.youtube.com/watch?v=4kcvIYYl
1,05:30:51,dupe,dupe: https://www.youtube.com/watch?v=wo34nBHM5sg,cluster=12 canonical=https://www.youtube.com/watch?v=4kcvIYYl
2,05:30:51,dupe,dupe: https://www.youtube.com/watch?v=dIAVt91Eu9o,cluster=12 canonical=https://www.youtube.com/watch?v=4kcvIYYl
3,05:30:51,dupe,dupe: https://www.reddit.com/r/movies/comments/1195irc/i_rewa,cluster=10 canonical=https://www.reddit.com/r/movies/comments
4,05:30:51,dupe,dupe: https://www.nytimes.com/2002/05/24/movies/film-review-a,cluster=9 canonical=https://www.nytimes.com/2002/05/24/movie
5,05:30:51,dupe,dupe: https://www.facebook.com/groups/166563453973502/posts/1,cluster=8 canonical=https://www.facebook.com/groups/16656345
6,05:30:51,dupe,dupe: https://www.facebook.com/groups/166563453973502/posts/1,cluster=8 canonical=https://www.facebook.com/groups/16656345
7,05:30:51,dupe,dupe: https://www.helpingwritersbecomeauthors.com/movie-story,cluster=7 canonical=https://www.helpingwritersbecomeauthors.
8,05:30:51,dupe,dupe: https://www.helpingwritersbecomeauthors.com/movie-story,cluster=7 canonical=https://www.helpingwritersbecomeauthors.
9,05:30:51,dupe,dupe: https://www.imdb.com/title/tt0278504/plotsummary/,cluster=6 canonical=https://www.imdb.com/title/tt0278504/plo



-- REFLECTION LOG --


,ts,agent,issue,fix
0,05:30:53,relevance,Prompt updated: Consider using more specific keywords or query phrases to fi...,old=d41d8cd9 new=3df58aa7
1,05:30:53,eval,Some results may be film analysis or reviews instead of the actual film,logged


## Cell 17 — Export to JSON

In [18]:
output = {
    'user_query':      USER_QUERY,
    'user_region':     USER_REGION,
    'run_id':          RUN_ID,
    'model':           REPLICATE_MODEL,
    'run_at':          time.strftime('%Y-%m-%d %H:%M:%S'),
    'run_metrics':     final_state.get('run_metrics', {}),
    'total_archived':  final_state.get('final_count', 0),
    'eval_accuracy':   final_state.get('eval_result', {}).get('accuracy_estimate'),
    'urls':            memory.get_archived_urls(run_id=RUN_ID),
    'eval_issues':     final_state.get('eval_result', {}).get('issues', []),
    'recommendations': final_state.get('eval_result', {}).get('recommendations', []),
}

with open('results.json', 'w') as f:
    json.dump(output, f, indent=2)

print('Saved to results.json')
print(json.dumps(output, indent=2)[:2000])

Saved to results.json
{
  "user_query": "INSOMNIA by Christopher Nolan",
  "user_region": "",
  "run_id": "6bada1e3-6f47-4cf3-a1e1-378128941e56",
  "model": "meta/meta-llama-3-8b-instruct",
  "run_at": "2026-03-25 05:31:59",
  "run_metrics": {
    "fetched_count": 131,
    "content_passed_count": 118,
    "relevant_count": 67,
    "total_records": 67,
    "clustered_count": 67,
    "singleton_count": 7,
    "dupe_count": 35,
    "merged_into_existing": 6,
    "paired_leftovers": 0,
    "canonical_candidates": 32,
    "total_clusters": 20,
    "archived_count": 67
  },
  "total_archived": 67,
  "eval_accuracy": 0.9,
  "urls": [
    {
      "url": "https://www.ultimatemovierankings.com/insomnia-2002/",
      "title": "Insomnia (2002) | Ultimate Movie Rankings",
      "score": 0.85,
      "category": "movie information",
      "keywords": [
        "Insomnia",
        "Christopher Nolan",
        "Al Pacino",
        "Robin Williams",
        "Hilary Swank"
      ],
      "cluster_id": 19

## Cell 18 — Download outputs

In [ ]:
from google.colab import files
import os

for fname in ['results.json', 'pipeline.db', 'human_review.csv', 'clustered_urls.csv']:
    if os.path.exists(fname):
        files.download(fname)
        print(f'Downloading {fname}')
    else:
        print(f'  {fname} not found (skipping)')


In [19]:
import json
import pandas as pd

with open('results.json', 'r') as f:
    results_data = json.load(f)

urls = results_data.get('urls', [])
if not urls:
    print('No URL records found in results.json')
else:
    rows = []
    for url_record in urls:
        rec = url_record.copy()
        rec['cluster_id'] = rec.get('cluster_id', 0)
        rows.append(rec)

    df_clustered = pd.DataFrame(rows)

    columns = [
        'cluster_id', 'url', 'title', 'score', 'category', 'keywords',
        'is_duplicate', 'query_source', 'run_id', 'user_query'
    ]
    for col in df_clustered.columns:
        if col not in columns:
            columns.append(col)
    df_clustered = df_clustered[columns]

    df_clustered = df_clustered.sort_values(by=['cluster_id', 'score'], ascending=[True, False])

    csv_name = f"clustered_urls_{results_data.get('run_id', 'run')[:8]}.csv"
    df_clustered.to_csv(csv_name, index=False)
    print(f"Grouped URLs saved to '{csv_name}'")

    print('First 10 rows of the clustered DataFrame:')
    display(df_clustered.head(10))

Grouped URLs saved to 'clustered_urls_6bada1e3.csv'
First 10 rows of the clustered DataFrame:


,cluster_id,url,title,score,category,keywords,is_duplicate,query_source,run_id,user_query,canonical_url,run_at
4,0,https://variety.com/2002/film/awards/insomnia-1117877182/,Insomnia,0.85,film,"[Christopher Nolan, Insomnia, film, awards, nominations]",False,Christopher Nolan's Insomnia film awards and nominations,6bada1e3-6f47-4cf3-a1e1-378128941e56,INSOMNIA by Christopher Nolan,https://variety.com/2002/film/awards/insomnia-1117877182/,2026-03-25 05:30:53
38,0,https://www.reddit.com/r/FIlm/comments/1gvvs4o/anyone_seen_christopher_nolan...,Anyone seen Christopher Nolan's Insomnia (2002)? : r/FIlm - Reddit,0.85,film,"[Christopher Nolan, Insomnia, Al Pacino]",False,Christopher Nolan's 2002 film Insomnia,6bada1e3-6f47-4cf3-a1e1-378128941e56,INSOMNIA by Christopher Nolan,,2026-03-25 05:30:53
42,0,https://www.reddit.com/r/FIlm/comments/1gvvs4o/anyone_seen_christopher_nolan...,Anyone seen Christopher Nolan's Insomnia (2002)? : r/FIlm,0.85,movie,"[Insomnia, Christopher Nolan, Al Pacino, Robin Williams]",False,Insomnia movie directed by Christopher Nolan,6bada1e3-6f47-4cf3-a1e1-378128941e56,INSOMNIA by Christopher Nolan,,2026-03-25 05:30:53
51,0,https://www.reddit.com/r/FIlm/comments/1gvvs4o/anyone_seen_christopher_nolan...,Anyone seen Christopher Nolan's Insomnia (2002)? : r/FIlm,0.85,film,"[Christopher Nolan, Insomnia, remake]",False,Christopher Nolan's film Insomnia,6bada1e3-6f47-4cf3-a1e1-378128941e56,INSOMNIA by Christopher Nolan,,2026-03-25 05:30:53
6,1,https://en.wikipedia.org/wiki/Insomnia_(2002_film),Insomnia (2002 film) - Wikipedia,0.85,film,"[Christopher Nolan, Insomnia, film, 2002, Wikipedia]",True,Christopher Nolan's Insomnia film awards and nominations,6bada1e3-6f47-4cf3-a1e1-378128941e56,INSOMNIA by Christopher Nolan,https://en.wikipedia.org/wiki/Insomnia_(2002_film),2026-03-25 05:30:53
8,1,https://en.wikipedia.org/wiki/Insomnia_(2002_film),Insomnia (2002 film) - Wikipedia,0.85,film,"[Insomnia, Christopher Nolan, 2002 film]",True,Insomnia movie by Christopher Nolan trailer,6bada1e3-6f47-4cf3-a1e1-378128941e56,INSOMNIA by Christopher Nolan,https://en.wikipedia.org/wiki/Insomnia_(2002_film),2026-03-25 05:30:53
18,1,https://en.wikipedia.org/wiki/Insomnia_(2002_film),Insomnia (2002 film) - Wikipedia,0.85,film,"[Christopher Nolan, Insomnia, film, 2002, Wikipedia]",True,Christopher Nolan's Insomnia film plot summary,6bada1e3-6f47-4cf3-a1e1-378128941e56,INSOMNIA by Christopher Nolan,https://en.wikipedia.org/wiki/Insomnia_(2002_film),2026-03-25 05:30:53
22,1,https://en.wikipedia.org/wiki/Insomnia_(2002_film),Insomnia (2002 film) - Wikipedia,0.85,film,"[Insomnia, Christopher Nolan, movie, film, 2002]",True,Insomnia movie directed by Christopher Nolan cast,6bada1e3-6f47-4cf3-a1e1-378128941e56,INSOMNIA by Christopher Nolan,https://en.wikipedia.org/wiki/Insomnia_(2002_film),2026-03-25 05:30:53
39,1,https://en.wikipedia.org/wiki/Insomnia_(2002_film),Insomnia (2002 film) - Wikipedia,0.85,film,"[Christopher Nolan, Insomnia, 2002 film]",True,Christopher Nolan's 2002 film Insomnia,6bada1e3-6f47-4cf3-a1e1-378128941e56,INSOMNIA by Christopher Nolan,https://en.wikipedia.org/wiki/Insomnia_(2002_film),2026-03-25 05:30:53
44,1,https://en.wikipedia.org/wiki/Insomnia_(2002_film),Insomnia (2002 film) - Wikipedia,0.85,film,"[Insomnia, Christopher Nolan, film, movie, 2002]",True,Insomnia movie directed by Christopher Nolan,6bada1e3-6f47-4cf3-a1e1-378128941e56,INSOMNIA by Christopher Nolan,https://en.wikipedia.org/wiki/Insomnia_(2002_film),2026-03-25 05:30:53
